In [ ]:
import math # có hàm check bằng nhau để tránh sai số dấu phẩy động
from itertools import permutations
from collections import deque
import time

1. Dùng thuật toán tìm kiếm cổ điển

In [ ]:
# DFS - Depth-First Search
def dfs(state):
    # Nếu state chỉ còn 1 phần tử, kiểm tra xem giá trị có bằng 24 không
    if len(state) == 1:
        val, expr = state[0]
        # Sử dụng math.isclose để tránh sai số của số thực (float)
        if math.isclose(val, 24, rel_tol=1e-5):
            return expr
        return None

    # Lồng 2 vòng lặp để lấy ra tất cả các hoán vị
    for i in range(len(state)):
        for j in range(len(state)):
            if i == j: # Không chọn cùng một số
                continue

            val1, expr1 = state[i]
            val2, expr2 = state[j]

            # Tạo danh sách các số còn lại sau khi rút val1 và val2 ra
            next_state_base = [state[k] for k in range(len(state)) if k != i and k != j]

            # Định nghĩa 4 phép toán: +, -, *, /
            operations = [
              (val1 + val2, f"({expr1} + {expr2})"),
              (val1 - val2, f"({expr1} - {expr2})"),
              (val1 * val2, f"({expr1} * {expr2})")
            ]
            # Kiểm tra điều kiện chia cho 0
            if val2 != 0:
              operations.append((val1 / val2, f"({expr1} / {expr2})"))

            # Thử lần lượt từng kết quả phép toán, ghép vào state mới và đi tiếp (DFS)
            for next_val, next_expr in operations:
              next_state = next_state_base + [(next_val, next_expr)]

              # Gọi đệ quy đi sâu xuống tầng tiếp theo
              result = dfs(next_state)

              # Nếu nhánh này trả về kết quả thành công, lập tức trả về (dừng duyệt)
              if result is not None:
                return result

    # Trả về None nếu duyệt hết các nhánh mà không có kết quả
    return None

In [ ]:
# BFS - Breadth-First Search
def bfs(state):
    queue = deque([state])

    # Duyệt khi hàng đợi vẫn còn trạng thái
    while queue:
        # Lấy trạng thái từ đầu hàng đợi ra để xử lý
        current_state = queue.popleft()

        # Kiểm tra trạng thái đích
        if len(current_state) == 1:
            val, expr = current_state[0]
            if math.isclose(val, 24, rel_tol=1e-5):
                return expr
            # Nếu chỉ còn 1 số nhưng không phải 24, bỏ qua nhánh này
            continue

        # Duyệt 2 vòng for để sinh ra tất cả hoán vị của 2 số
        n = len(current_state)
        for i in range(n):
            for j in range(n):
                if i == j: # Không chọn cùng một số
                    continue

                val1, expr1 = current_state[i]
                val2, expr2 = current_state[j]

                # Giữ lại các phần tử không tham gia vào phép tính lần này
                next_state_base = [current_state[k] for k in range(n) if k != i and k != j]

                # Tính toán 4 phép toán
                operations = [
                    (val1 + val2, f"({expr1} + {expr2})"),
                    (val1 - val2, f"({expr1} - {expr2})"),
                    (val1 * val2, f"({expr1} * {expr2})")
                ]
                if val2 != 0:
                    operations.append((val1 / val2, f"({expr1} / {expr2})"))

                # Tạo trạng thái mới và đẩy vào CUỐI hàng đợi
                for next_val, next_expr in operations:
                    next_state = next_state_base + [(next_val, next_expr)]
                    queue.append(next_state)

    # Nếu vòng lặp kết thúc mà queue rỗng (duyệt hết cây) thì không có đáp án
    return None

2. Dùng ToT

In [ ]:
# tải thư viện mỗi lần mở project chạy lại
# !pip install groq
!pip install -q -U openai

2.1. Llama 17B và các hàm cần thiết

In [ ]:
# import Llama và các thư viện cần thiết
from google.colab import userdata
from groq import Groq

# Lấy API key từ khoá bí mật của Colab
api_key_17b = userdata.get('llama_api_key')

# Khởi tạo client dùng Llama
client_17b = Groq(api_key=api_key_17b)

In [ ]:
# Hàm sinh ra trạng thái của Llama
def llama_generate_thoughts(state_str):
  system_prompt = """Generate 5 possible arithmetic operations (+, -, *, /) using exactly 2 numbers from the given input.
Format strictly as: [Equation] (left: [Remaining numbers]). Do not output any other text."""

  # Nạp 2 ví dụ (Few-shot) bằng cấu trúc hội thoại giúp model nhỏ thẩm thấu tốt hơn
  messages = [
    {"role": "system", "content": system_prompt},
    # Ví dụ 1
    {"role": "user", "content": "Input: 4 9 10 13"},
    {"role": "assistant", "content": "10 - 4 = 6 (left: 6 9 13)\n9 + 4 = 13 (left: 10 13 13)\n10 / 4 = 2.5 (left: 2.5 9 13)\n13 * 4 = 52 (left: 9 10 52)\n13 - 10 = 3 (left: 3 4 9)"},
    # Ví dụ 2
    {"role": "user", "content": "Input: 1 2 3 4"},
    {"role": "assistant", "content": "1 + 2 = 3 (left: 3 3 4)\n4 - 3 = 1 (left: 1 1 2)\n4 * 2 = 8 (left: 1 3 8)\n3 / 1 = 3 (left: 2 3 4)\n4 + 1 = 5 (left: 2 3 5)"},

    {"role": "user", "content": f"Input: {state_str}"}
  ]

  try:
    response = client_17b.chat.completions.create(
      model="meta-llama/llama-4-scout-17b-16e-instruct",
      messages=messages,
      # temperature 0.7 để nó chịu khó nghĩ ra nhiều phép tính đa dạng
      temperature=0.7,
      max_completion_tokens=200,
      stream=False
    )

    output_text = response.choices[0].message.content.strip()
    thoughts = [line.strip() for line in output_text.split('\n') if line.strip()]

    return thoughts

  except Exception as e:
    print(f"Lỗi khi gọi API Generate: {e}")
    return []

In [ ]:
# Hàm đánh giá trạng thái của Llama
def llama_evaluate_state(state_str):
    # System prompt: CHỈ ĐƯỢC NHẢ RA 1 CHỮ
    system_prompt = "Classify if the given numbers can combine to make 24 using +, -, *, /. Reply with exactly ONE word: 'sure', 'likely', or 'impossible'. Do not explain."

    # Bơm đầy đủ 3 trường hợp để nó load đủ pattern
    messages = [
        {"role": "system", "content": system_prompt},
        # Nhìn là biết ra 24 -> sure
        {"role": "user", "content": "Input: 4 6"},
        {"role": "assistant", "content": "sure"},
        {"role": "user", "content": "Input: 8 3"},
        {"role": "assistant", "content": "sure"},

        # Nhìn vô vọng -> impossible
        {"role": "user", "content": "Input: 1 1 1"},
        {"role": "assistant", "content": "impossible"},
        {"role": "user", "content": "Input: 0 5 9"},
        {"role": "assistant", "content": "impossible"},

        # Nhìn có vẻ tiềm năng nhưng chưa chắc -> likely
        {"role": "user", "content": "Input: 4 9 10 13"},
        {"role": "assistant", "content": "likely"},
        {"role": "user", "content": "Input: 2 4 8"},
        {"role": "assistant", "content": "likely"},

        {"role": "user", "content": f"Input: {state_str}"}
    ]

    try:
        response = client_17b.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=messages,
            temperature=0.0, # Tuyệt đối giữ 0.0 để câu trả lời dứt khoát hơn
            max_completion_tokens=10,
            stream=False
        )

        # Kiểm tra lỡ LLM nói nhiều
        result = response.choices[0].message.content.strip().lower()
        if "sure" in result: return "sure"
        elif "likely" in result: return "likely"
        else: return "impossible"

    except Exception as e:
        print(f"Lỗi khi gọi API Eval: {e}")
        return "impossible" # Gặp lỗi thì vứt nhánh đó đi

2.2.  Llama 70B và các hàm cần thiết

In [ ]:
from groq import Groq
from google.colab import userdata

# Khai báo client theo chuẩn Groq
client_70b = Groq(api_key=userdata.get('groq_api_key'))
client_70b2 = Groq(api_key=userdata.get('groq2_api_key'))

clients = [client_70b, client_70b2] # mảng chứa 2 client
cur_client = 0 # index của client hiện tại

In [ ]:
# Hàm sinh ra trạng thái tiếp theo
def groq_generate_thoughts(state_str):
    global cur_client
    prompt = f"""You are solving the Game of 24. Propose up to 5 valid operations (+, -, *, /) using 2 numbers from the Input.
Rules:
- Fractions/decimals are allowed.
- NO extra text. Format STRICTLY as shown.

Input: 4 9 10 13
Output:
10 - 4 = 6 (left: 6 9 13)
9 + 4 = 13 (left: 10 13 13)
10 / 4 = 2.5 (left: 2.5 9 13)
13 * 4 = 52 (left: 9 10 52)

Input: {state_str}
"""

    for attempt in range(2):
        try:
            time.sleep(2.5)
            response = clients[cur_client].chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                stream=False
            )
            return response.choices[0].message.content.strip().split('\n')

        except Exception as e:
            error_msg = str(e).lower()
            # Nếu bắt được lỗi 429 (Rate Limit) -> Tiến hành đổi key
            if "429" in error_msg or "rate limit" in error_msg:
                print(f"Key {cur_client + 1} hết token! Đang tự động đổi sang Key {2 - cur_client}...")
                cur_client = 1 - cur_client # Đổi 0 thành 1, đổi 1 thành 0
            else:
                print(f"Lỗi khác Generate: {e}")
                break # Lỗi khác thì dẹp luôn không thử lại

    return []

In [ ]:

# Hàm đánh giá trạng thái
def groq_evaluate_state(state_str):
    global cur_client
    prompt = f"""Evaluate if the Input numbers can combine to make 24 using +, -, *, /. Fractions are allowed.
Reply ONLY with one word: sure, likely, or impossible.

Input: 4 6
Output: sure
Input: 1 1 1
Output: impossible
Input: 4 9 10 13
Output: likely
Input: 8 3
Output: sure

Input: {state_str}
Output:"""

    for attempt in range(2):
        try:
            time.sleep(2)
            response = clients[cur_client].chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                stream=False
            )
            result = response.choices[0].message.content.strip().lower()

            if "sure" in result: return "sure"
            elif "likely" in result: return "likely"
            else: return "impossible"
        except Exception as e:
            error_msg = str(e).lower()
            if "429" in error_msg or "rate limit" in error_msg:
                print(f"Key {cur_client + 1} hết token! Đang tự động đổi sang Key {2 - cur_client}...")
                cur_client = 1 - cur_client
            else:
                break

    return "impossible"

2.3. Các hàm dùng chung

In [ ]:
# Hàm tổng quát sinh ra phép toán bằng LLM tương ứng truyền vào
def llm_generate_thoughts(state_str, model_name):
  if model_name == "llama17b":
    return llama_generate_thoughts(state_str)
  elif model_name == "llama70b":
    return groq_generate_thoughts(state_str)
  else:
    raise ValueError(f"Lỗi: Model type '{model_type}' chưa được hỗ trợ!")

In [ ]:
# Hàm đánh giá trạng thái hiện tại có kết quả không bằng LLM tương ứng truyền vào
def llm_evaluate_state(state_str, model_name):
  if model_name == "llama17b":
    return llama_evaluate_state(state_str)
  elif model_name == "llama70b":
    return groq_evaluate_state(state_str)
  else:
    raise ValueError(f"Lỗi: Model type '{model_type}' chưa được hỗ trợ!")

In [ ]:
# Hàm thực hiện bước sinh trạng thái kế tiếp và đánh giá
def get_tot_successors(state_str, model_type):
  valid_successors = []
  old_nums = [float(x) for x in state_str.split()] # Đếm số lượng số hiện tại
  # Gọi LLM sinh ý tưởng (thêm sleep để tránh rate limit)
  time.sleep(2)
  thoughts = llm_generate_thoughts(state_str, model_type)

  for thought in thoughts:
    # 2. Tách ra thành chuỗi equation và chuỗi tập hợp
    try:
      # Xử lý trường hợp LLM sinh thiếu chữ "left"
      if "(left:" not in thought:
        continue
      equation_part, left_part = thought.split("(left:")
      equation = equation_part.strip()
      next_state_str = left_part.replace(")", "").strip()

      # Kiểm tra số lượng số mới có giảm hay ko (LLM có bịa thêm số hay ko)
      new_nums = [float(x) for x in next_state_str.split()]
      if len(new_nums) != len(old_nums) - 1:
        continue
      if "=" not in equation:
        continue
      left_expr, right_val = equation.split("=")
      # Bắt buộc vế trái phải có dấu phép tính (+, -, *, /)
      if not any(op in left_expr for op in ['+', '-', '*', '/']):
        continue
      # Kiểm tra vế trái có bằng vế phải không
      if abs(eval(left_expr.strip()) - float(right_val.strip())) > 1e-5:
        continue # Nếu tính sai thì bỏ qua
    except (ValueError, ZeroDivisionError, SyntaxError):
      continue # Bỏ qua nếu LLM sinh sai format

    # Gọi LLM chấm điểm và bỏ các node impossible
    time.sleep(2)
    score = llm_evaluate_state(next_state_str, model_type)
    # Nếu là sure hoặc likely thì thêm vào
    if score in ['sure', 'likely']:
      valid_successors.append((next_state_str, equation))

  return valid_successors # trả về các trạng thái phù hợp (chuỗi trạng thái, phép toán)

2.4. Thuật toán tìm kiếm cho ToT

In [ ]:
# ToT dùng DFS
def tot_dfs(initial_state_str, model_type):
  # Dùng stack
    stack = [(initial_state_str, [])]

    max_steps = 100
    steps = 0

    while stack and steps < max_steps:
        steps += 1

        # Lấy trạng thái trên đỉnh stack
        current_state_str, history = stack.pop()
        # Kiểm tra đã đến goal chưa
        try:
            nums = [float(x) for x in current_state_str.split()]
            if len(nums) == 1:
                import math
                if math.isclose(nums[0], 24, rel_tol=1e-5):
                    return " -> ".join(history) # THÀNH CÔNG!
                continue
        except ValueError:
            continue

        # Lấy các trạng thái mới hợp lệ
        successors = get_tot_successors(current_state_str, model_type)

        for next_state_str, equation in successors:
            # Ghi lại lịch sử có hiển thị số còn lại
            step_description = f"{equation} [còn lại: {next_state_str}]"
            stack.append((next_state_str, history + [step_description]))

    # Duyệt hết mà không ra kết quả
    return None

In [ ]:
from collections import deque
import math


# BFS quá tốn token vì duyệt quá nhiều node nên em chuyển qua dùng DFS =))


# ToT dùng BFS
def tot_bfs(initial_state_str, model_type):
  queue = deque([(initial_state_str, [])])
  max_steps = 100
  steps = 0

  while queue and steps < max_steps:
    steps += 1
    current_state_str, history = queue.popleft()

    # Kiểm tra trạng thái hiện tại có phải goal ko
    try:
      # Tách chuỗi thành mảng các con số
      nums = [float(x) for x in current_state_str.split()]
      if len(nums) == 1: # Kiểm tra xem có còn 1 số không
        if math.isclose(nums[0], 24, rel_tol=1e-5):
          return " -> ".join(history)
        # không phải 24 thì bỏ qua
        continue
    except ValueError:
      # Nếu LLM trả về ko đúng format -> Bỏ qua nhánh này
      continue

    # Lấy các trạng thái kế tiếp từ LLM
    successors = get_tot_successors(current_state_str, model_type)
    # Đẩy vào hàng đợi
    for next_state_str, equation in successors:
      queue.append((next_state_str, history + [equation]))
  # Trả về None nếu duyệt hết mà không thấy
  return None

In [ ]:
# Hàm giải game of 24 bằng ToT
def solve_game_of_24_tot(numbers, model_name):
    # Biến mảng [4, 9, 10, 13] thành chuỗi "4 9 10 13"
    initial_state_str = " ".join(map(str, numbers))

    # Gọi thuật toán ToT DFS
    solution = tot_dfs(initial_state_str, model_name)

    # In kết quả
    if solution:
        return f"{solution}"
    else:
        return "Không có lời giải (Hoặc LLM không tìm ra)"

In [ ]:
# Hàm giải game of 24 sử dụng tìm kiếm cổ điển
def solve_game_of_24(numbers, method: str):
  # trạng thái bắt đầu lưu trữ các tuple (float, str) dùng cho việc tính toán và xuất chuỗi
  initial_state = [(float(num), str(num)) for num in numbers]

  if method == "dfs":
    solution = dfs(initial_state)
  elif method == "bfs":
    solution = bfs(initial_state)
  else:
    raise ValueError("Không tìm thấy phương pháp")

  # trả về kết quả
  if solution:
    # xoá 2 dấu ngoặc ngoài cùng
    if solution.startswith('(') and solution.endswith(')'):
      solution = solution[1:-1]
    return f"{solution} = 24"
  else:
    return "Không có lời giải"


2.5. Input - Output Prompting với 2 model Llama 17B và 70B

In [ ]:
# Hàm giải Game of 24 bằng Input-Output Prompting
def solve_game_of_24_iop(numbers, model_name):
    global cur_client
    nums_str = " ".join(map(str, numbers))

    prompt = f"""Task: Solve the Game of 24.
Use EXACTLY the four given numbers and basic arithmetic (+, -, *, /) to create an expression that equals 24.

CRITICAL RULES:
1. You MUST use each number exactly once.
2. FORMAT RULE: Output ONLY the mathematical expression. Do NOT write "= 24". Do NOT write explanations. NO extra text.
3. If you cannot find a solution, output exactly the word: impossible

Examples:
Input: 4 7 8 8
Output: 8 * ((4 + 7) - 8)

Input: 1 1 1 1
Output: impossible

Input: 8 3 8 3
Output: 8 / (3 - (8 / 3))

Input: {nums_str}
Output:"""

    for attempt in range(2):
        try:
            time.sleep(3) # Phanh câu giờ chống spam
            response = clients[cur_client].chat.completions.create(
                model=model_name,
                messages=[
                    # Dùng System Prompt để doạ nó
                    {"role": "system", "content": "You are a calculator. You output strictly math expressions or the word 'impossible'. Nothing else."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                max_tokens=20, # Ép token cực thấp để chống lảm nhảm
                stream=False
            )

            # Làm sạch output
            raw_result = response.choices[0].message.content.strip()
            result = raw_result.split('\n')[0].strip()
            result = result.replace("Output:", "").replace("= 24", "").strip()

            return result

        except Exception as e:
            error_msg = str(e).lower()
            if "429" in error_msg or "rate limit" in error_msg:
                print(f"Key {cur_client + 1} đạt giới hạn! Đổi sang Key {2 - cur_client}...")
                cur_client = 1 - cur_client # <--- Đã cập nhật phép toán xoay vòng
            else:
                return f"Lỗi: {e}"
    return "impossible"

In [ ]:
test_cases = [
    [4, 7, 8, 8],    # ví dụ
    [4, 9, 10, 13],  # ví dụ
    [1, 1, 1, 1],    # Không có lời giải
    [0, 9, 3, 3],    # Trung bình
    [3, 3, 8, 8],    # Cực khó
    [1, 2, 3, 4],    # Cực dễ
    [6, 6, 6, 6],    # Dễ
    [2, 5, 6, 8],    # Trung bình
    [3, 5, 7, 9],    # Trung bình (đòi hỏi phép nhân 2 cụm rồi trừ cho nhau)
    [1, 5, 5, 5]     # Cực khó (phải sinh ra phân số ở bước trung gian)
]
method = "bfs"
print(f"Kết quả sử dụng tìm kiếm cổ điển bằng thuật toán {method}: ")
for nums in test_cases:
  ans = solve_game_of_24(nums, method)
  print(f"{nums}: {ans}")

models = ['llama70b', 'llama17b']
for input_model in models:
  print(f"Kết quả sử dụng ToT bằng mô hình {input_model}")
  for nums in test_cases:
    ans = solve_game_of_24_tot(nums, input_model)
    print(f"{nums}: {ans}")


Kết quả sử dụng tìm kiếm cổ điển bằng thuật toán bfs: 
[4, 7, 8, 8]: 8 * ((4 + 7) - 8) = 24
[4, 9, 10, 13]: (4 - 10) * (9 - 13) = 24
[1, 1, 1, 1]: Không có lời giải
[0, 9, 3, 3]: (3 * (0 + 9)) - 3 = 24
[3, 3, 8, 8]: 8 / (3 - (8 / 3)) = 24
[1, 2, 3, 4]: 4 * (3 + (1 + 2)) = 24
[6, 6, 6, 6]: (6 + 6) + (6 + 6) = 24
[2, 5, 6, 8]: 8 * (6 + (2 - 5)) = 24
[3, 5, 7, 9]: (3 + 5) + (7 + 9) = 24
[1, 5, 5, 5]: 5 * (5 - (1 / 5)) = 24
Kết quả sử dụng ToT bằng mô hình llama70b
[4, 7, 8, 8]: 8 - 4 = 4 [còn lại: 7 8 4] -> 7 - 4 = 3 [còn lại: 8 3] -> 8 * 3 = 24 [còn lại: 24]
[4, 9, 10, 13]: 13 - 9 = 4 [còn lại: 4 4 10] -> 10 - 4 = 6 [còn lại: 4 6] -> 4 * 6 = 24 [còn lại: 24]
[1, 1, 1, 1]: Không có lời giải (Hoặc LLM không tìm ra)
[0, 9, 3, 3]: 0 + 3 = 3 [còn lại: 3 9 3] -> 3 * 9 = 27 [còn lại: 3 27] -> 27 - 3 = 24 [còn lại: 24]
[3, 3, 8, 8]: Không có lời giải (Hoặc LLM không tìm ra)
[1, 2, 3, 4]: 1 + 2 = 3 [còn lại: 3 4 4] -> 4 + 4 = 8 [còn lại: 3 8] -> 8 * 3 = 24 [còn lại: 24]
[6, 6, 6, 6]: 6 * 6 = 36 [

In [ ]:
test_cases = [
    [4, 7, 8, 8],    # ví dụ
    [4, 9, 10, 13],  # ví dụ
    [1, 1, 1, 1],    # Không có lời giải
    [0, 9, 3, 3],    # Trung bình
    [3, 3, 8, 8],    # Cực khó
    [1, 2, 3, 4],    # Cực dễ
    [6, 6, 6, 6],    # Dễ
    [2, 5, 6, 8],    # Trung bình
    [3, 5, 7, 9],    # Trung bình (đòi hỏi phép nhân 2 cụm rồi trừ cho nhau)
    [1, 5, 5, 5]     # Cực khó (phải sinh ra phân số ở bước trung gian)
]
models = ['llama-3.3-70b-versatile', 'meta-llama/llama-4-scout-17b-16e-instruct']
for input_model in models:
  print(f"Kết quả sử dụng Input-Output Prompting bằng mô hình {input_model}")
  for nums in test_cases:
    ans = solve_game_of_24_iop(nums, input_model)
    print(f"{nums}: {ans}")

Kết quả sử dụng Input-Output Prompting bằng mô hình llama-3.3-70b-versatile
[4, 7, 8, 8]: 8 * (7 - (8 / 4))
[4, 9, 10, 13]: (13 - 9) * (10 / 4)
[1, 1, 1, 1]: impossible
[0, 9, 3, 3]: (3 * 3) + (9 / 0)
[3, 3, 8, 8]: (8 * 3) + (8 / 3)
[1, 2, 3, 4]: (3 + 4 + 1) * 2
[6, 6, 6, 6]: (6 * 6) - (6 / 6)
[2, 5, 6, 8]: (8 * 3) + (5 + 2)
[3, 5, 7, 9]: (9 * 7) - (5 + 3)
[1, 5, 5, 5]: (5 * 5) - (5 / 1)
Kết quả sử dụng Input-Output Prompting bằng mô hình meta-llama/llama-4-scout-17b-16e-instruct
[4, 7, 8, 8]: 8 * (7 - (8 / 8))
[4, 9, 10, 13]: 10 + 13 + 4 - 9 *  4 + 13 + 10
[1, 1, 1, 1]: impossible
[0, 9, 3, 3]: impossible
[3, 3, 8, 8]: 8 / (3 - 8/3)
[1, 2, 3, 4]: (1 + 2) * 3 * 4
[6, 6, 6, 6]: 6 + 6 + 6 + 6
[2, 5, 6, 8]: (8 - 5) * (6 + 2)
[3, 5, 7, 9]: (9 - 5) * (7 - 3)
[1, 5, 5, 5]: 5 * (5 - (5 / 5))
